# RAG Pipeline — Fully on Cloudflare

Build a complete Retrieval-Augmented Generation pipeline using only Cloudflare services:

```
Browser Run (crawl) → Workers AI (embed) → Vectorize (store) → Workers AI (query)
```

No Pinecone. No OpenAI embeddings. No external vector database. Everything runs on Cloudflare's infrastructure.

**Prerequisites:**
- API token with: **Workers AI → Read**, **Browser Rendering → Edit**, **Vectorize → Edit**
- A Vectorize index (instructions below)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

assert os.getenv("CLOUDFLARE_ACCOUNT_ID")
assert os.getenv("CLOUDFLARE_API_TOKEN")

## Create a Vectorize index

Run this once in your terminal:

```bash
npx wrangler vectorize create my-knowledge-base --dimensions 768 --metric cosine
```

This creates a vector index that stores 768-dimensional embeddings (matching `bge-base-en-v1.5`).

## Step 1: Ingest — Crawl web pages

In [ ]:
from pydantic_ai_cloudflare import BrowserRunToolset

browser = BrowserRunToolset(tools=["browse"])

# Load some documentation pages
pages = [
    "https://developers.cloudflare.com/workers-ai/",
    "https://developers.cloudflare.com/vectorize/",
    "https://developers.cloudflare.com/browser-run/",
]

documents = []
for url in pages:
    content = await browser._browse(url)
    documents.append({"text": content, "source": url})
    print(f"Loaded: {url} ({len(content)} chars)")

print(f"\nTotal: {len(documents)} documents")

## Step 2: Embed + Store — Index in Vectorize

In [ ]:
from pydantic_ai_cloudflare import VectorizeToolset

# Replace with your index name
knowledge = VectorizeToolset(index_name="my-knowledge-base")

for doc in documents:
    # Store first 2000 chars of each page
    result = await knowledge._store(
        doc["text"][:2000],
        source=doc["source"],
    )
    print(result)

## Step 3: Query — Search and answer

In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent

class Answer(BaseModel):
    response: str
    sources: list[str]

agent = Agent(
    "openai:@cf/meta/llama-3.3-70b-instruct-fp8-fast",
    output_type=Answer,
    toolsets=[knowledge],
    system_prompt=(
        "Answer questions using the search_knowledge tool. "
        "Always cite which sources you used."
    ),
)

result = agent.run_sync("What embedding models does Cloudflare offer?")

print(f"Answer: {result.output.response}")
print(f"Sources: {result.output.sources}")

## The full picture

What just happened:

1. **Browser Run** rendered 3 JS-heavy documentation pages and returned markdown
2. **Workers AI** (`bge-base-en-v1.5`) generated 768-dim embeddings for each document
3. **Vectorize** stored the vectors with source metadata
4. On query: **Workers AI** embedded the question → **Vectorize** found relevant docs → **Workers AI** (Llama 3.3) generated the answer

All on Cloudflare. Free tier. No external services.